# TorchCodec：先短測，再分階段測完整影片

使用 T4 GPU runtime、Python 3.13+、PyTorch 2.11+ 與相容 torchvision。第一階段只跑 baseline／stage1 各 180 幀；stage2 和完整影片均需另外開啟。保留原 FPS、完整輸出幀及 FFmpeg 音訊尾端修正。這份 notebook 不會移除 torchvision，也不會安裝 torchaudio。

**執行前**修改影片與權重路徑。固定 `RUN_NAME` 可續作；只跳過完整驗證成功的整組，不能接續影片中途的模型狀態。CPU 測試不能證明 T4 加速。

In [ ]:
import sys, subprocess, json
from pathlib import Path
def checked_capture(command):
    completed = subprocess.run(command, text=True, capture_output=True)
    if completed.returncode:
        print(completed.stdout, end="")
        print(completed.stderr, end="", file=sys.stderr)
        completed.check_returncode()
    return completed
if sys.version_info < (3, 13):
    raise RuntimeError("需要 Python 3.13+，請選相容的 Colab runtime")
check = checked_capture([sys.executable, "-c", """
import json, torch, torchvision
from packaging.version import Version
assert Version(torch.__version__.split("+")[0]) >= Version("2.11"), torch.__version__
assert torch.cuda.is_available(), "請選 GPU runtime；不允許 CPU fallback"
assert torch.version.cuda == "13.0", "此安裝 cell 僅支援已安裝的 cu130；不自動替換 PyTorch"
print(json.dumps({"torch": torch.__version__, "torchvision": torchvision.__version__,
                  "cuda_build": torch.version.cuda, "gpu": torch.cuda.get_device_name(0)}))
"""])
accelerators = json.loads(check.stdout)
print(accelerators)


取得專案並安裝相容套件。保留既有 torch／torchvision build；TorchCodec 僅安裝 0.17.0 cu130 wheel。若這個 session 已匯入 cv2、torch 等套件，安裝後重新啟動 session，再執行設定與測試 cells。套件檢查使用獨立 Python 程序。

In [ ]:
repo = Path("/content/auto-vertical-reframe-torchcodec")
if not repo.exists():
    subprocess.run(["git", "clone", "--branch", "main",
                    "https://github.com/delvilo/auto-vertical-reframe.git", str(repo)], check=True)
else:
    # 保留既有修改；不執行 reset 或覆寫 notebook。
    dirty = subprocess.check_output(["git", "status", "--porcelain"], cwd=repo, text=True)
    if dirty.strip():
        raise RuntimeError("checkout 有未提交修改，請先保留修改再更新：" + dirty)
    subprocess.run(["git", "pull", "--ff-only"], cwd=repo, check=True)
subprocess.run([sys.executable, str(repo / "install_colab.py")], cwd=repo, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "--force-reinstall",
                "torchcodec==0.17.0", "--index-url", "https://download.pytorch.org/whl/cu130"],
               check=True)
verify = checked_capture([sys.executable, "-c", """
import json, shutil, torch, torchvision, torchcodec
assert shutil.which("ffmpeg") and shutil.which("ffprobe"), "需要 FFmpeg / ffprobe CLI"
# import torchcodec 同時檢查其 FFmpeg shared libraries 能否載入。
from torchcodec.decoders import VideoDecoder
boxes = torch.tensor([[0., 0., 2., 2.]], device="cuda")
scores = torch.tensor([1.], device="cuda")
assert torchvision.ops.nms(boxes, scores, 0.5).numel() == 1
print(json.dumps({"torch": torch.__version__, "torchvision": torchvision.__version__,
                  "torchcodec": torchcodec.__version__}))
"""])
installed = json.loads(verify.stdout)
for name in ("torch", "torchvision"):
    assert installed[name] == accelerators[name], (name, installed[name], accelerators[name])
print(installed)


設定資料與固定續作目錄。`RESTORE_FROM_DRIVE=True` 只用於本機目錄不存在時，從 Drive 還原整個資料夾（含 MP4）。下載權重會使用 Drive 上的 Torch cache；精簡後的 DeepGaze 不需要 548 MB ImageNet VGG 權重。

In [ ]:
from google.colab import drive
import os, shutil, sys, subprocess, json
from pathlib import Path
drive.mount("/content/drive")
repo = Path("/content/auto-vertical-reframe-torchcodec")
RUN_NAME = "torchcodec-t4-20261012-v1"  # 請固定名稱，勿每次加新時間
SOURCE = Path("/content/drive/MyDrive/crop/video.mp4")  # 修改
SEG_MODEL = Path("/content/drive/MyDrive/crop/yolo26n-seg.pt")
POSE_MODEL = Path("/content/drive/MyDrive/crop/yolo26n-pose.pt")
OUTPUT_DIR = Path("/content/reframe-torchcodec") / RUN_NAME
BACKUP_DIR = Path("/content/drive/MyDrive/reframe-benchmarks") / RUN_NAME
RESTORE_FROM_DRIVE = False
TRACKING_PROFILE = "balanced"  # conservative / balanced / fast
SEG_MAX_GAP = 1  # 資料流對照固定 1；方案 B 測試改為 None，採 profile 預設
if RESTORE_FROM_DRIVE:
    if OUTPUT_DIR.exists():
        raise RuntimeError("本機目錄已存在，避免覆蓋；請關閉 RESTORE_FROM_DRIVE")
    shutil.copytree(BACKUP_DIR, OUTPUT_DIR)
for path in (SOURCE, SEG_MODEL, POSE_MODEL):
    if not path.is_file():
        raise FileNotFoundError(path)
local_source = Path("/content") / (RUN_NAME + SOURCE.suffix)
shutil.copy2(SOURCE, local_source)
os.environ["TORCH_HOME"] = "/content/drive/MyDrive/reframe-model-cache/torch"
common = [sys.executable, "-u", str(repo / "scripts/colab_torchcodec.py"),
          "--input", str(local_source), "--seg-model", str(SEG_MODEL),
          "--pose-model", str(POSE_MODEL), "--output-dir", str(OUTPUT_DIR),
          "--backup-dir", str(BACKUP_DIR), "--device", "0",
          "--saliency-device", "cuda", "--decode-device", "cuda:0",
          "--video-encoder", "hevc_nvenc", "--decode-batch-size", "4",
          "--tracking-profile", TRACKING_PROFILE]
if SEG_MAX_GAP is not None:
    common += ["--seg-max-gap", str(SEG_MAX_GAP)]
print("本機結果：", OUTPUT_DIR)
print("Drive 備份：", BACKUP_DIR)


第一階段：baseline 與 NVDEC reference 各 180 幀。沒有額外暖機或重複長片。收集器會驗證實際 NVDEC、模型裝置、NVENC、FPS、幀數與音訊終點，逐組備份；失敗保留原始 traceback 和已完成的 Segment。

In [ ]:
subprocess.run(common + ["--profiles", "baseline", "stage1", "--max-frames", "180"],
               cwd=repo, check=True)


第二階段：將 `RUN_STAGE2` 改為 `True`。GPU tensor 路徑可能因插值差異改變像素與模型結果，需比較成片；短片未呼叫 DeepGaze 時會顯示 `not_exercised`，不算完成其 GPU 路徑驗證。

In [ ]:
RUN_STAGE2 = False
if RUN_STAGE2:
    subprocess.run(common + ["--profiles", "stage2", "--max-frames", "180"], cwd=repo, check=True)
else:
    print("stage2 未執行；第一階段確認後將 RUN_STAGE2 設為 True")


完整影片：確認短測與畫面後，將 `RUN_FULL=True`，一次選一組。要求同一影片、模型、程式、套件與設定先有成功的 180 幀短測。每組可能耗時很久，建議逐組執行並查看 Drive 備份；runtime 硬性回收時 finally 無法保證執行。

In [ ]:
RUN_FULL = False
FULL_PROFILE = "stage1"  # baseline / stage1 / stage2
if RUN_FULL:
    if FULL_PROFILE not in {"baseline", "stage1", "stage2"}:
        raise ValueError(FULL_PROFILE)
    subprocess.run(common + ["--profiles", FULL_PROFILE, "--full-video"], cwd=repo, check=True)
else:
    print("完整影片未執行；需明確開啟 RUN_FULL")


閱讀最近一次結果。`record.json` 保留每次嘗試；每五秒／鏡頭分段在 `run.segments.csv/jsonl`，可比較 YOLO 次數與預測幀。`diagnostics.zip` 不含 MP4，適合上傳分析；成片另存於本機與 Drive。GPU 非同步 host 計時不能直接當作 kernel 時間。

In [ ]:
results = json.loads((OUTPUT_DIR / "results.json").read_text())
print("狀態：", results["status"])
for run in results["runs"]:
    summary = run.get("summary", {})
    print(json.dumps({"profile": run["profile"], "status": run["status"],
        "frames": summary.get("frames_processed"),
        "elapsed_seconds": summary.get("elapsed_seconds"),
        "devices": run.get("device_validation"), "decoder": run.get("decoder_validation"),
        "video_io": summary.get("video_io"), "directory": run["directory"]},
        ensure_ascii=False, indent=2))
print("可上傳分析：", OUTPUT_DIR / "diagnostics.zip")
print("Drive：", BACKUP_DIR)
# 需要下載到電腦時，取消下面兩行註解：
# from google.colab import files
# files.download(str(OUTPUT_DIR / "diagnostics.zip"))
